# La chaîne de traitement, étape par étape

**Le code réel, à chaque étape, suivi de sa démonstration**

Tassilo Westphalen · CPES Sciences des données, Arts et Cultures — PSL / Louis-le-Grand

---

Ce notebook accompagne l'étude sur l'identité de Ziak
([`02_stylometrie_ziak.ipynb`](02_stylometrie_ziak.ipynb)). Il ne refait pas
l'analyse : il montre **comment elle est faite**, fonction par fonction.

Le code affiché n'est pas recopié : il est **extrait des modules à l'exécution**
par `inspect.getsource`. Un extrait recopié se désynchronise du jour où l'on
touche au module, et personne ne s'en aperçoit. Extrait, il est nécessairement
celui qui tourne.

## Plan

1. Mise en place
2. Récolte — d'où viennent les données
3. Prétraitement — du texte aux nombres
4. Balises — qui chante quelle strophe
5. Moteur d'attribution — mesurer une distance
6. Une attribution complète, de bout en bout
7. Les analyses et la restitution

> **Les démonstrations de nettoyage portent sur un texte inventé**, jamais sur
> des paroles réelles : le mécanisme se montre aussi bien, sans reproduire
> d'œuvre protégée.

## 1. Mise en place

Un utilitaire d'abord : `source()` affiche le code d'une fonction, avec la
coloration syntaxique. C'est lui qui garantit que ce notebook montre le code
réel et non sa paraphrase.

In [1]:
import inspect
import re
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import Code, display

import genius_sections
import stylo_attribution
import stylo_features


def source(fonction) -> Code:
    """Le code réel d'une fonction, extrait du module à l'exécution."""
    return Code(inspect.getsource(fonction), language="python")


EXPORT = Path("export")
print("modules chargés :", stylo_features.__name__, "·",
      stylo_attribution.__name__, "·", genius_sections.__name__)

modules chargés : stylo_features · stylo_attribution · genius_sections


## 2. Récolte

### 2.1 D'où viennent les données

Trois sources, et elles n'ont pas les mêmes propriétés.

| Source | Ce qu'elle donne | Volume |
|---|---|---|
| Hugging Face — jeu LRFAF | le corpus publié, **paroles sans balises** | 37 307 titres |
| genius.com, par artiste | Mikeysem, web7, Ziak, avec balises | 3 artistes |
| genius.com, par URL | les balises de tout le corpus | 34 666 pages |

La troisième ligne est celle qui a coûté six heures, et la section 4 dira
pourquoi elle était nécessaire.

### 2.2 Le corpus publié se charge, se filtre, se met en cache

`corpus_csv()` choisit le corpus selon la variante demandée — le publié, ou
l'un des deux nettoyés. C'est ce qui permet de rejouer l'étude trois fois sans
rien écraser.

In [2]:
source(stylo_features.corpus_csv)

def corpus_csv() -> Path:
    """Chemin du corpus à analyser, selon la variante active."""
    if VARIANTE in FICHIERS_VARIANTE:
        p = Path(FICHIERS_VARIANTE[VARIANTE])
        if p.exists():
            return p
        raise FileNotFoundError(
            f"{p} absent — lancer 19_collecte_corpus.py puis "
            "20_corpus_sans_invites.py.")
    # Le dépôt amont a renommé le corpus `corpus.csv` ; l'ancien nom reste
    # accepté pour les copies locales déjà téléchargées.
    for nom in ("corpus.csv", "RapFr.csv"):
        if Path(nom).exists():
            return Path(nom)
    raise FileNotFoundError(
        "Corpus LRFAF introuvable — le télécharger depuis "
        "huggingface.co/datasets/regicid/LRFAF (voir README).")

### 2.3 La collecte par artiste

Pour les artistes absents du corpus. L'API de Genius ne sert pas les paroles —
elles sont licenciées séparément — donc les métadonnées viennent de l'endpoint
JSON et le texte de la page HTML. C'est la méthode qu'emploie `lyricsgenius`,
et celle qu'a employée LRFAF lui-même.

### 2.4 La re-collecte des balises

Le cœur du problème traité en section 4 : le corpus publié a perdu ses balises
de section. Elles sont encore sur Genius, dont le corpus conserve l'URL de
chaque morceau.

In [3]:
import importlib.util

spec = importlib.util.spec_from_file_location("collecte", "19_collecte_corpus.py")
collecte = importlib.util.module_from_spec(spec)
spec.loader.exec_module(collecte)

display(source(collecte.paroles))

def paroles(url: str) -> tuple[str | None, str]:
    """Bloc de paroles balisé d'une page Genius, ou (None, motif)."""
    s = session()
    for essai in range(MAX_ESSAIS):
        try:
            r = s.get(url, timeout=30)
        except requests.RequestException:
            time.sleep(PAUSE_ERREUR * (essai + 1))
            continue
        if r.status_code == 404:
            return None, "404"
        if r.status_code in (429, 403) or r.status_code >= 500:
            # Genius limite le débit : on attend de plus en plus longtemps.
            time.sleep(PAUSE_ERREUR * (2 ** essai) + random.uniform(0, 5))
            continue
        soup = BeautifulSoup(
            r.text.replace("<br/>", "\n").replace("<br>", "\n"), "html.parser")
        # `data-lyrics-container` est l'attribut stable, que Genius maintient
        # pour son propre front-end. Les deux replis visent les balisages plus
        # anciens, dont les noms de classes minifiés changent à chaque refonte
        # (c'est sur eux que repose lyricsgenius, et ce qui le casse).
        divs = soup.find_all("div", attrs={"data-lyrics-container": "true"})
        if not divs:
            divs = soup.find_all("div", class_=re.compile(r"Lyrics__Container"))
        if not divs:
            divs = soup.find_all("div", class_=re.compile(r"^Lyrics-\w{2}.\w+.[1]"))
        if not divs:
            return None, "pas de bloc de paroles"
        return "\n".join(d.get_text() for d in divs), "ok"
    return None, "refus répété"

Trois détails de ce code méritent d'être relevés.

- **Le sélecteur** vise d'abord `data-lyrics-container`, l'attribut stable que
  Genius maintient pour son propre front-end. Les deux replis couvrent les
  balisages plus anciens, dont les noms de classes minifiés changent à chaque
  refonte — c'est sur eux que repose `lyricsgenius`, et ce qui le casse
  périodiquement. Sur 34 666 pages, zéro échec de parsing.
- **Le repli exponentiel** sur 429 / 403 / 5xx : la collecte dure six heures,
  un refus passager ne doit pas la faire échouer.
- **Le 404 n'est pas réessayé** : c'est une page supprimée, pas un incident.
  Il y en a eu 66.

## 3. Prétraitement — du texte aux nombres

Quatre opérations, chacune montrée puis démontrée. La démonstration porte sur
un texte inventé pour l'occasion.

In [4]:
# Texte d'exemple, écrit pour ce notebook — jamais des paroles réelles.
EXEMPLE = "J'ai vu l'aube, puis j'suis rentré... (uh) 100 % réveillé !"
print(EXEMPLE)

J'ai vu l'aube, puis j'suis rentré... (uh) 100 % réveillé !


### 3.1 Nettoyer

Deux caractères sont **délibérément conservés** : les accents, qui portent du
signal en français, et l'apostrophe, qui porte l'élision — donc une partie de
la grammaire.

In [5]:
source(stylo_features.clean_lyrics)

def clean_lyrics(text: str) -> str:
    """Normalise un texte de chanson : minuscules, apostrophes, ponctuation."""
    if not isinstance(text, str):
        return ""
    text = GENIUS_NOISE.sub(" ", text)
    text = text.translate(APOSTROPHES).lower()
    text = text.replace("\n", " ")
    text = NON_LETTERS.sub(" ", text)
    return re.sub(r"\s+", " ", text).strip()

In [6]:
propre = stylo_features.clean_lyrics(EXEMPLE)
print("avant :", EXEMPLE)
print("après :", propre)

avant : J'ai vu l'aube, puis j'suis rentré... (uh) 100 % réveillé !
après : j'ai vu l'aube puis j'suis rentré uh réveillé


### 3.2 Découper en mots

L'apostrophe finale est capturée **avec** le mot : `j'ai` donne `j'` puis `ai`.
Autrement dit `je` et `j'` deviennent deux traits distincts.

Ce choix, qui paraît anodin, est ce qui a permis de démasquer un faux marqueur :
Ziak semblait sous-employer « je » d'un facteur 5, alors qu'il compense
entièrement en « j' ».

In [7]:
source(stylo_features.tokenize)

def tokenize(text: str) -> list[str]:
    """Découpe en mots, l'apostrophe séparant l'élision (j'ai -> j' + ai)."""
    return re.findall(r"[a-zàâäéèêëïîôöùûÿçœæ]+'?", text)

In [8]:
mots = stylo_features.tokenize(propre)
print(mots)
print(f"\n{len(mots)} tokens — noter « j' » isolé de « ai »")

["j'", 'ai', 'vu', "l'", 'aube', 'puis', "j'", 'suis', 'rentré', 'uh', 'réveillé']

11 tokens — noter « j' » isolé de « ai »


### 3.3 Les 4-grammes de caractères

Fenêtre glissante de quatre caractères, **espaces compris**. Les espaces
comptent : ils capturent les fins et débuts de mots, donc une part de la
syntaxe, sans passer par le lexique. C'est ce qui rend ces traits robustes au
sujet abordé.

In [9]:
source(stylo_features.char_ngrams)

def char_ngrams(text: str, n: int = 4) -> list[str]:
    """4-grammes de caractères, espaces compris (capture les fins de mots)."""
    return [text[i : i + n] for i in range(len(text) - n + 1)]

In [10]:
grammes = stylo_features.char_ngrams(propre)
print(grammes[:12])
print(f"\n{len(grammes)} 4-grammes pour {len(propre)} caractères")

["j'ai", "'ai ", 'ai v', 'i vu', ' vu ', 'vu l', "u l'", " l'a", "l'au", "'aub", 'aube', 'ube ']

42 4-grammes pour 45 caractères


### 3.4 Filtrer le corpus

Deux filtres, et le second compte plus qu'il n'y paraît : un même morceau publié
sous deux artistes attribuerait le même texte à deux auteurs, ce qui fausse
mécaniquement les distances.

In [11]:
source(stylo_features.load_corpus)

def load_corpus(min_tokens: int = 100) -> pd.DataFrame:
    """Charge le corpus LRFAF, nettoie les paroles et retire doublons et titres courts.

    Les doublons de paroles entre artistes (0,3 % du corpus) correspondent à des
    featurings ou à des rééditions : les conserver reviendrait à attribuer le
    même texte à deux auteurs, ce qui fausse mécaniquement les distances.
    """
    df = pd.read_csv(corpus_csv())
    df = df.dropna(subset=["artist", "lyrics", "year"]).copy()
    df["lyrics_clean"] = df["lyrics"].map(clean_lyrics)
    df["tokens"] = df["lyrics_clean"].map(tokenize)
    df["n_tokens"] = df["tokens"].map(len)
    df = df[df["n_tokens"] >= min_tokens].copy()

    # Un même texte peut apparaître sous plusieurs artistes (featurings).
    df["dup_key"] = df["lyrics_clean"].str.slice(0, 300)
    df = df.drop_duplicates(subset="dup_key", keep="first")

    df["year"] = df["year"].astype(int)
    return df.reset_index(drop=True)

### 3.5 Mettre en cache

Le choix structurant de tout le projet tient dans une ligne de cette fonction :
les comptes sont stockés **par chanson**, jamais par artiste.

Un document d'artiste de n'importe quelle taille se recompose ensuite par simple
addition de lignes. Sans cela, chaque rééchantillonnage exigerait de
revectoriser le corpus — et le protocole, qui en compte des centaines, serait
inexécutable.

In [12]:
cache = stylo_features.build_cache()
meta = cache["meta"]

print("le cache contient :")
for cle, valeur in cache.items():
    forme = getattr(valeur, "shape", None) or f"{len(valeur)} entrées"
    print(f"  {cle:18} {forme}")

le cache contient :
  meta               (32923, 6)
  counts_mfw         (32923, 500)
  counts_char        (32923, 3000)
  counts_word_ext    (32923, 6000)
  vocab_mfw          500 entrées
  vocab_char         3000 entrées
  vocab_word_ext     6000 entrées


## 4. Balises — qui chante quelle strophe

### 4.1 Le problème

Sur Genius, chaque bloc de paroles porte le nom de celui qui le chante. **LRFAF
a supprimé ces balises avant publication** : sur 400 textes tirés au hasard,
aucun n'en porte.

Or un couplet d'invité est écrit par l'invité. Le laisser dans le corpus de
l'artiste principal rapproche mécaniquement cet artiste de tous ceux qu'il a
invités — et l'hypothèse testée porte justement sur une proximité.

### 4.2 Lire une balise

`interpretes()` renvoie les interprètes nommés par une balise, ou `None` si elle
n'en nomme aucun.

In [13]:
source(genius_sections.interpretes)

def interpretes(balise: str) -> set[str] | None:
    """Interprètes nommés par une balise ; None si elle n'en nomme aucun."""
    contenu = re.sub(r"\([^)]*\)", "", balise).strip()      # « (x2) », « (Ziak) »…
    tiret = re.match(r"^([^:\-–—]+?)\s*[-–—]\s*(.+)$", contenu)
    if ":" in contenu:
        gauche, noms = contenu.split(":", 1)
        # Ordre inversé : « [Soprano : Outro] », « [Lefa : Couplet 1] ».
        if not _est_libelle(gauche) and _libelle_seul(noms):
            noms = gauche
    elif tiret and _est_libelle(tiret.group(1)):
        # Genius emploie le tiret aussi bien que le deux-points :
        # « [Couplet 3 - Kool Shen] ». Le libellé doit précéder, sinon
        # « Pre-refrain » ou « Jay-Z » seraient coupés en deux.
        noms = tiret.group(2)
    elif _est_libelle(contenu):
        return None
    else:
        noms = contenu                                        # « [Sofiane] »
    # « [Refrain - Outro] », « [Pont : Instrumental] » : deux libellés, aucun nom.
    out = {_norme(n) for n in SEPARATEURS.split(noms)
           if _norme(n) and not _libelle_seul(n)}
    return out or None

L'ordre des branches est le correctif d'un bug réel : Genius emploie le
deux-points **et** le tiret, et seul le premier était lu, si bien que
`[Couplet 3 - Kool Shen]` passait pour une section de l'artiste principal.

Mais le tiret ne peut être accepté que si un libellé de section le précède —
sinon `Pre-refrain` et `Jay-Z` seraient coupés en deux. D'où la table de cas
ci-dessous, qui sert de test.

In [14]:
cas = ["Couplet 1", "Refrain", "Couplet 2 : Akhenaton",
       "Couplet 3 - Kool Shen & Joey Starr", "Pre-refrain",
       "Post-refrain : Ziak", "Jay-Z", "Sofiane", "?"]
pd.DataFrame({
    "balise": [f"[{c}]" for c in cas],
    "interprètes nommés": [genius_sections.interpretes(c) or "— (artiste principal)"
                           for c in cas],
})

,balise,interprètes nommés
0,[Couplet 1],— (artiste principal)
1,[Refrain],— (artiste principal)
2,[Couplet 2 : Akhenaton],{akhenaton}
3,[Couplet 3 - Kool Shen & Joey Starr],"{joeystarr, koolshen}"
4,[Pre-refrain],— (artiste principal)
5,[Post-refrain : Ziak],{ziak}
6,[Jay-Z],{jayz}
7,[Sofiane],{sofiane}
8,[?],— (artiste principal)


### 4.3 Retirer les strophes d'invités

La règle est volontairement stricte : une section n'est conservée que si elle
n'est attribuée à personne, ou au seul artiste principal. **Une section partagée
est retirée**, faute de pouvoir savoir qui en a écrit quoi.

In [15]:
source(genius_sections.retire_featurings)

def retire_featurings(texte: str, alias_principal: set[str]) -> tuple[str, dict]:
    """Paroles réduites aux sections de l'artiste principal, et statistiques.

    `alias_principal` : noms sous lesquels l'artiste principal peut apparaître
    dans les balises (« web7 », « 7 Jaws », « 7Jaws »…).
    """
    alias = {_norme(a) for a in alias_principal}
    garde, n_garde, n_retire, n_sections = [], 0, 0, 0
    courant_garde = True
    for ligne in texte.split("\n"):
        if est_balise_de_section(ligne):
            contenu = LIGNE_BALISE.match(ligne).group(1)
            if _norme(contenu).startswith("parolesde"):
                continue                                      # métadonnée Genius
            noms = interpretes(contenu)
            courant_garde = noms is None or noms <= alias
            if courant_garde:
                garde.append(ligne)
            else:
                n_sections += 1
            continue
        n = len(re.findall(r"\w+", ligne))
        if courant_garde:
            garde.append(ligne)
            n_garde += n
        else:
            n_retire += n
    return "\n".join(garde), {"mots_gardes": n_garde, "mots_retires": n_retire,
                              "sections_invites": n_sections}

In [16]:
# Morceau fictif, écrit pour ce notebook.
MORCEAU = """[Couplet 1]
première ligne de l'artiste principal
deuxième ligne de l'artiste principal

[Couplet 2 : Invité]
ligne écrite par quelqu'un d'autre
seconde ligne du même invité

[Refrain]
refrain non attribué, donc gardé"""

garde, stats = genius_sections.retire_featurings(MORCEAU, {"Principal"})
print(garde)
print("\n→", stats)

[Couplet 1]
première ligne de l'artiste principal
deuxième ligne de l'artiste principal

[Refrain]
refrain non attribué, donc gardé

→ {'mots_gardes': 17, 'mots_retires': 12, 'sections_invites': 1}


### 4.4 Membre de groupe ou invité de passage ?

C'est la difficulté du nettoyage. La règle naïve viderait les groupes de leur
contenu : chez IAM, les blocs nomment Akhenaton et Shurik'n, dont les couplets
*sont* le texte d'IAM.

C'est la **fréquence** qui les sépare — un nom qui revient dans une grande
partie des morceaux d'un artiste est celui d'un membre, et ses couplets sont
gardés ; un nom qui n'apparaît qu'une ou deux fois est celui d'un invité, et
ses couplets sont retirés.

In [17]:
spec = importlib.util.spec_from_file_location("nettoyage", "20_corpus_sans_invites.py")
nettoyage = importlib.util.module_from_spec(spec)
spec.loader.exec_module(nettoyage)

display(source(nettoyage.membres_par_artiste))

def membres_par_artiste(collecte: dict, urls: pd.DataFrame) -> tuple[dict, pd.DataFrame]:
    """Pour chaque artiste, les intervenants récurrents — réputés internes."""
    par_artiste = defaultdict(Counter)
    n_titres = Counter()
    for url, artiste in zip(urls.url, urls.artist):
        r = collecte.get(url)
        if r is None:
            continue
        n_titres[artiste] += 1
        par_artiste[artiste].update(intervenants(P.strip_genius_header(r["lyrics_raw"])))

    membres, lignes = {}, []
    for artiste, compte in par_artiste.items():
        n = n_titres[artiste]
        internes = {_norme(artiste)}
        for nom, k in compte.items():
            if not n:
                continue
            part = k / n
            motif = ("répertoire" if part >= PART_MEMBRE else
                     "récurrent" if part >= PART_RECURRENT and k >= TITRES_RECURRENT
                     else None)
            if motif:
                internes.add(nom)
                lignes.append({"artiste": artiste, "intervenant": nom, "titres": k,
                               "sur": n, "part": part, "motif": motif})
        membres[artiste] = internes
    # L'intervenant départage les ex æquo : sans lui, l'ordre suivrait celui
    # d'un ensemble Python, qui change d'une exécution à l'autre.
    return membres, pd.DataFrame(lignes).sort_values(["artiste", "part", "intervenant"],
                                                     ascending=[True, False, True])

Le résultat de cette règle, artiste par artiste, est publié plutôt que caché
dans le code — voici ce qu'elle a retenu pour quelques groupes.

In [18]:
membres = pd.read_csv(EXPORT / "20_2_membres_detectes.csv")
apercu = membres[membres.artiste.isin(["IAM", "Suprême NTM", "113"])]
apercu[["artiste", "intervenant", "titres", "sur", "part", "motif"]]

,artiste,intervenant,titres,sur,part,motif
3,113,rimk,65,80,0.812500,répertoire
4,113,ap,60,80,0.750000,répertoire
5,113,mokobe,21,80,0.262500,récurrent
6,113,rohff,7,80,0.087500,récurrent
7,113,ogb,5,80,0.062500,récurrent
8,113,manukey,4,80,0.050000,récurrent
247,IAM,akhenaton,190,273,0.695971,répertoire
248,IAM,shurikn,185,273,0.677656,répertoire
249,IAM,freeman,34,273,0.124542,récurrent
659,Suprême NTM,koolshen,46,63,0.730159,répertoire


**Cette règle a une limite, et elle est connue.** Freeman figure sur 6,7 % des
morceaux d'IAM dont il est membre ; Rohff sur 6,3 % de ceux de 113 dont il est
seulement invité. Aucun seuil ne les sépare.

Les deux erreurs possibles n'étant pas symétriques, le réglage penche vers la
conservation : garder le couplet d'un invité ne fait que reproduire le défaut du
corpus d'origine, tandis que retirer celui d'un membre en fabriquerait un
nouveau. Les 12 % de mots retirés sont donc un **plancher**, pas une valeur
exacte.

## 5. Moteur d'attribution — mesurer une distance

Quatre fonctions, dans l'ordre où elles s'enchaînent.

### 5.1 Échantillonner à taille contrôlée

On échantillonne des **chansons entières**, pas des mots : découper au milieu
d'un morceau mélangerait des contextes. Sans cette contrainte de taille, la
distance mesure surtout la couverture de vocabulaire, et un artiste prolifique
paraît proche de tout le monde.

In [19]:
source(stylo_attribution.sample_indices)

def sample_indices(song_ids: np.ndarray, n_tokens: np.ndarray, target: int,
                   rng: np.random.Generator) -> np.ndarray:
    """Tire des chansons sans remise jusqu'à atteindre `target` tokens."""
    order = rng.permutation(len(song_ids))
    cumsum = np.cumsum(n_tokens[order])
    k = int(np.searchsorted(cumsum, target) + 1)
    k = min(k, len(order))
    return song_ids[order[:k]]

### 5.2 Agréger en fréquences

In [20]:
source(stylo_attribution.make_docs)

def make_docs(counts: sparse.csr_matrix, groups: dict[str, np.ndarray]
              ) -> tuple[list[str], np.ndarray]:
    """Somme les comptes par groupe, puis convertit en fréquences relatives."""
    names = list(groups)
    mat = np.vstack([
        np.asarray(counts[groups[name]].sum(axis=0)).ravel() for name in names
    ])
    totals = mat.sum(axis=1, keepdims=True)
    totals[totals == 0] = 1.0
    return names, mat / totals

### 5.3 Standardiser, puis mesurer

Le détail qui compte tient en deux lignes : **les z-scores sont estimés sur les
candidats seuls**, et la requête est projetée dans ce référentiel. C'est la
situation réelle d'un texte anonyme confronté à un corpus connu — inclure la
requête dans le calcul de la moyenne la ferait participer à sa propre
normalisation.

In [21]:
source(stylo_attribution.rank_candidates)

def rank_candidates(query_freq: np.ndarray, cand_freqs: np.ndarray,
                    metric: str = "cosine_delta") -> np.ndarray:
    """Distances requête -> candidats, standardisées sur le même corpus.

    Les z-scores sont estimés sur les candidats seuls : la requête est projetée
    dans ce référentiel, comme un texte anonyme confronté à un corpus connu.
    """
    mu = cand_freqs.mean(axis=0)
    sd = cand_freqs.std(axis=0, ddof=0)
    sd = np.where(sd < 1e-12, 1.0, sd)
    cand_z = (cand_freqs - mu) / sd
    query_z = (query_freq - mu) / sd
    return DISTANCES[metric](query_z, cand_z)

### 5.4 Le score de séparation

La fonction la plus importante du projet, et la plus courte.

Un classement désigne **toujours** un premier : ce n'est pas une information. La
question utile est de savoir de combien il se détache. Ce z-score négatif est la
seule grandeur comparable d'une requête à l'autre — donc la seule qui permette
de confronter Ziak à des cas dont on connaît la réponse.

In [22]:
source(stylo_attribution.separation_score)

def separation_score(dists: np.ndarray, best_idx: int) -> float:
    """À quel point le meilleur candidat se détache-t-il du reste ?

    Renvoie un z-score négatif : -3 signifie que le candidat retenu est à trois
    écarts-types sous la moyenne des distances aux autres candidats. C'est cette
    grandeur — et non la distance brute — qui est comparable d'une requête à
    l'autre, et qui permet de confronter le cas Ziak aux cas de contrôle.
    """
    others = np.delete(dists, best_idx)
    mu, sd = others.mean(), others.std(ddof=0)
    if sd < 1e-12:
        return 0.0
    return float((dists[best_idx] - mu) / sd)

## 6. Une attribution complète, de bout en bout

Toutes les pièces assemblées. Une quinzaine de lignes suffisent à poser la
question de l'étude : *qui, parmi les 392 candidats, écrit comme Ziak ?*

In [23]:
mat = cache["counts_char"]
n_tokens = meta["n_tokens"].to_numpy()
totaux = meta.groupby("artist")["n_tokens"].sum()

pool = sorted(a for a in totaux[totaux >= 12_000].index if a != "Ziak")
titres = {a: meta.index[meta["artist"] == a].to_numpy() for a in pool}
idx_ziak = meta.index[meta["artist"] == "Ziak"].to_numpy()
rng = np.random.default_rng(20260930)

# 1. chaque candidat ramené à 12 000 mots exactement
groupes = {a: stylo_attribution.sample_indices(
    titres[a], n_tokens[titres[a]], 12_000, rng) for a in pool}

# 2. agrégation en fréquences relatives
noms, freqs = stylo_attribution.make_docs(mat, groupes)

# 3. la requête : tout le corpus de Ziak
requete = np.asarray(mat[idx_ziak].sum(axis=0)).ravel()
requete = requete / requete.sum()

# 4. distances standardisées, puis séparation du premier
distances = stylo_attribution.rank_candidates(requete, freqs, "cosine_delta")
ordre = np.argsort(distances)
separation = stylo_attribution.separation_score(distances, int(ordre[0]))

classement = pd.DataFrame({"artiste": [noms[i] for i in ordre[:8]],
                           "distance": distances[ordre[:8]]})
print(f"{len(pool)} candidats · séparation du premier : {separation:.2f}")
classement

392 candidats · séparation du premier : -2.26


,artiste,distance
0,Beendo Z,0.813663
1,Zkr,0.815786
2,Fresh laDouille,0.819257
3,UZI (FRA),0.821693
4,Rimkus,0.828129
5,Maes,0.834829
6,Lemon Haze,0.835438
7,Kerchak,0.837929


Le premier du classement se détache de **moins de 3 écarts-types**. C'est peu :
un alias authentique produit une séparation autour de −4,6, et un auteur absent
du corpus autour de −3,0.

C'est tout le propos de la section 5.4 — sans ce repère, le tableau ci-dessus
désignerait un coupable avec assurance.

In [24]:
seuils = pd.read_csv(EXPORT / "03_3_seuils_decision.csv")
seuils = seuils[(seuils.features == "char") & (seuils.metric == "cosine_delta")]
print(f"séparation observée pour Ziak    : {separation:.2f}")
print(f"repère H₁ — auteur bien présent  : {seuils.sep_H1_correct_moy.iloc[0]:.2f}")
print(f"repère H₀ — auteur absent        : {seuils.sep_H0_moy.iloc[0]:.2f}")

séparation observée pour Ziak    : -2.26
repère H₁ — auteur bien présent  : -4.59
repère H₀ — auteur absent        : -3.05


## 7. Les analyses et la restitution

Le moteur ci-dessus est appelé par treize scripts, chacun répondant à une
question et écrivant sa réponse en CSV.

In [25]:
analyses = pd.DataFrame([
    ("02", "l'approche intuitive a-t-elle raison ?", "25 % contre 91,5 %"),
    ("03", "que vaut la méthode quand on connaît la réponse ?", "90 % au premier rang"),
    ("04", "un artiste du corpus porte-t-il la signature de Ziak ?", "non — H₀ 4 fois sur 4"),
    ("05", "le verdict résiste-t-il aux objections ?", "oui, sur les quatre testées"),
    ("06", "à quoi ressemble son écriture ?", "banal au centre, sans proche parent"),
    ("08-09", "Mikeysem au format LRFAF", "mikeysem_lrfaf.csv"),
    ("10", "Ziak est-il Mikeysem ?", "non — 58ᵉ sur 493"),
    ("13-14", "la validation tient-elle sur des cas réels ?", "Joke → Ateyaba 1ᵉʳ sur 393"),
    ("15-16", "web7 écrit-il ses textes ?", "non — mais co-auteur de 10 titres"),
    ("18", "les featurings faussent-ils le classement ?", "non — mesuré, pas supposé"),
    ("19-20", "récupérer et retirer les strophes d'invités", "deux corpus propres"),
], columns=["script", "question", "réponse"])
analyses.set_index("script")

,question,réponse
script,,
02,l'approche intuitive a-t-elle raison ?,"25 % contre 91,5 %"
03,que vaut la méthode quand on connaît la réponse ?,90 % au premier rang
04,un artiste du corpus porte-t-il la signature d...,non — H₀ 4 fois sur 4
05,le verdict résiste-t-il aux objections ?,"oui, sur les quatre testées"
06,à quoi ressemble son écriture ?,"banal au centre, sans proche parent"
08-09,Mikeysem au format LRFAF,mikeysem_lrfaf.csv
10,Ziak est-il Mikeysem ?,non — 58ᵉ sur 493
13-14,la validation tient-elle sur des cas réels ?,Joke → Ateyaba 1ᵉʳ sur 393
15-16,web7 écrit-il ses textes ?,non — mais co-auteur de 10 titres


L'ordre n'est pas décoratif : **la validation précède les tests**. Un résultat
négatif n'a de valeur que si l'on a d'abord établi que la méthode sait trouver
quand il y a quelque chose à trouver.

### La restitution

`07_figures.py` fabrique les figures depuis les CSV. `article_contenu.py` porte
le texte et **relit tous les chiffres dans les résultats au moment de la
génération** : aucune valeur n'est écrite à la main, donc aucune ne peut se
désynchroniser de l'analyse qu'elle cite. Deux moteurs partagent ce texte, l'un
vers le PDF, l'autre vers le Word.

### Tout enchaîner

```bash
./pipeline.sh                    # les huit étapes
./pipeline.sh --depuis variantes # à partir de celle-ci
./pipeline.sh balises --oui      # autorise l'étape longue (~6 h)
```

Chaque étape est idempotente : ce qui existe n'est pas refait, et une relance
reprend où elle s'est arrêtée.

---

**Pour aller plus loin** — [`02_stylometrie_ziak.ipynb`](02_stylometrie_ziak.ipynb)
présente l'étude et ses résultats ; [`METHODE.md`](METHODE.md) expose la
démarche ; [`CODE.md`](CODE.md) commente le code bloc par bloc ;
[`PIPELINE.md`](PIPELINE.md) en donne le schéma.